# Advanced Regression, Boosting Approaches

"Define a multiple regression task, i.e., using more than one input feature, and solve it using 2 advanced regression approaches (not linear).
Compare and evaluate the approaches using appropriate metrics. Discuss the result. "

Advanced models that we will try: (Deep) Neural Network, Gradient Boosting (i also did Hist+), XGBoost and EBM. (Other models mentioned: SVM and Random Forest Regressor).

The target variable has to be numerical, and in order to avoid data leakage, we will predict PCIAT-PCIAT_Total, which was eliminated before the missing value imputation. This variable has around 40% of missing values, so we will work only on the records with not missing PCIAT. Then, the obtained models could be used to predict these missing ones!
We will use all of the rest of the features as input features (except for sii, because they are too correlated, since we as explained before we believe that sii was extracted by discretizing PCIAT).

# Data Preparation & Partitioning

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
#df = pd.read_csv('../dataset/cmi_internet_regression.csv')
df = pd.read_csv('cmi_internet_regression.csv')

In [3]:
X = df.drop(columns=['PCIAT-PCIAT_Total', 'sii'])
y = df['PCIAT-PCIAT_Total'].astype(int)

num_cols = X.select_dtypes(include=np.number).columns.tolist()
print("Variables used:")
print(num_cols)
print("Shape:", X.shape)

Variables used:
['Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score', 'Physical-Height', 'Physical-Weight', 'Physical-Waist_Circumference', 'Physical-Diastolic_BP', 'Fitness_Endurance-Max_Stage', 'Physical-HeartRate', 'Physical-Systolic_BP', 'FGC-FGC_CU', 'FGC-FGC_GSND', 'FGC-FGC_GSD', 'FGC-FGC_PU', 'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL', 'BIA-BIA_DEE', 'BIA-BIA_Activity_Level_num', 'BIA-BIA_BMC', 'BIA-BIA_BMR', 'BIA-BIA_ECW', 'BIA-BIA_Fat', 'BIA-BIA_Frame_num', 'BIA-BIA_ICW', 'BIA-BIA_LDM', 'BIA-BIA_LST', 'BIA-BIA_SMM', 'PAQ_Total', 'SDS-SDS_Total_T', 'PreInt_EduHx-computerinternet_hoursday', 'Fitness_Endurance-Time']
Shape: (2468, 32)


In [4]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

X = pd.DataFrame(
    X,
    columns=num_cols,
    index=df.index
)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=100) # no stratifying because y is not categorical

# Gradient Boosting

In [6]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [8]:
# Using same structure as Guidotti's classification example, turning it to regression.

reg = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=1.0,
    max_depth=3,
    random_state=0
)

reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred))) # Measures prediction error magnitude in original units. The lower the better.
print('R2:', r2_score(y_test, y_pred)) # Measures variance of the target variable explained by the model. The closer to 1 the better.

MAE: 17.616823233576785
MSE: 498.7602033407251
RMSE: 22.332939872321447
R2: -0.22511365934762972


In [ ]:
reg = GradientBoostingRegressor() # OBS: much better results with default parameters...

reg.fit(X_train, y_train)

y_pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))

MAE: 13.742193248210782
MSE: 304.1729815081988
RMSE: 17.440555653653895
R2: 0.2528544339460519


In [9]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import RepeatedKFold

In [10]:
import warnings
warnings.simplefilter("ignore")

In [ ]:
# Doing hyper-parameter tuning:

model = GradientBoostingRegressor(random_state=0)

param_distributions = {
    "n_estimators": list(np.arange(50, 201, 10)),
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0],
    "max_depth": [2, 3, 5, 7, 9, 11],
    "subsample": [0.3, 0.7, 1.0], 
    "loss": ['squared_error', 'absolute_error']
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=100, # takes 13.75 minutes
    cv=RepeatedKFold(random_state=0),
    scoring="neg_mean_squared_error", # In sklearn higher scores are considered better, but smaller MSE is better, therefore internally maximize -MSE
    random_state=0,
    n_jobs=-1 # use all available CPU cores in parallel for the computation
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", search.best_params_)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))


Best params: {'subsample': 0.3, 'n_estimators': np.int64(70), 'max_depth': 2, 'loss': 'squared_error', 'learning_rate': 0.05}
MAE: 13.69716440228888
MSE: 296.64713555889205
RMSE: 17.22344726118706
R2: 0.2713403047290165


## Hist Gradient Boosting
It is a faster, more scalable version of gradient boosting for classification. It uses histogram binning of feature values, which makes training more efficient on larger datasets (>10.000)

In [13]:
from sklearn.ensemble import HistGradientBoostingRegressor

In [45]:
reg = HistGradientBoostingRegressor()

reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

print('MSE:', mean_squared_error(y_test, y_pred))
print('R2:', r2_score(y_test, y_pred))

MSE: 323.559466330727
R2: 0.20523506254526336


In [ ]:
model = HistGradientBoostingRegressor(random_state=0)

param_distributions = {
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0, 2.0],
    "max_depth": [2, 3, 5, 7, 9, 11],
    "loss": ['squared_error', 'absolute_error']
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=100, # takes 1.75 minutes
    cv=RepeatedKFold(random_state=0),
    scoring="neg_mean_squared_error",
    random_state=0,
    n_jobs=-1
)

search.fit(X_train, y_train)

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

print("Best params:", search.best_params_)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))

Best params: {'max_depth': 2, 'loss': 'squared_error', 'learning_rate': 0.05}
MAE: 13.598248342952019
MSE: 292.3673207750515
RMSE: 17.09875202390664
R2: 0.28185289076944486


# XGBoost
https://xgboost.readthedocs.io/en/stable/python/python_intro.html

In [16]:
# %pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 8.6 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [15]:
from xgboost import XGBRegressor

In [16]:
reg = XGBRegressor(
    objective='reg:squarederror',  # regression objective
    max_depth=6,
    learning_rate=1.0,
    gamma=0.0,
    reg_lambda=1,
    tree_method='exact',           # or 'approx'
    random_state=42
)

reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

print('MSE:', mean_squared_error(y_test, y_pred))
print('R2:', r2_score(y_test, y_pred))


MSE: 503.8026428222656
R2: -0.23749947547912598


In [44]:
reg = XGBRegressor()

reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

print('MSE:', mean_squared_error(y_test, y_pred))
print('R2:', r2_score(y_test, y_pred))

MSE: 347.06573486328125
R2: 0.14749616384506226


In [ ]:
reg = XGBRegressor(objective='reg:squarederror', random_state=42)

param_distributions = {
    "n_estimators": list(np.arange(50, 301, 20)),
    "max_depth": [2, 3, 5, 7, 9, 11],
    "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0, 2.0],
    "gamma": [0, 0.05, 0.1, 0.5, 1.0],
    "reg_lambda": [0.1, 0.5, 1.0, 2.0, 5.0],
}

search = RandomizedSearchCV(
    estimator=reg,
    param_distributions=param_distributions,
    n_iter=100, # takes 7.25 minutes
    cv=RepeatedKFold(random_state=0),
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    random_state=0
)

search.fit(X_train, y_train)

best_reg = search.best_estimator_
y_pred = best_reg.predict(X_test)

print("Best params:", search.best_params_)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))


Best params: {'reg_lambda': 0.1, 'n_estimators': np.int64(110), 'max_depth': 2, 'learning_rate': 0.05, 'gamma': 0.05}
MAE: 13.579619407653809
MSE: 292.0215759277344
RMSE: 17.08863879680691
R2: 0.28270214796066284


# LightGBM
https://lightgbm.readthedocs.io/en/latest/Python-Intro.html

In [ ]:
# %pip install lightgbm
# or use conda install -c conda-forge lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 1.7 MB/s  0:00:01 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [33]:
from lightgbm import LGBMRegressor

In [ ]:
# Base model
reg = LGBMRegressor(random_state=42)  
 
reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000406 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5162
[LightGBM] [Info] Number of data points in the train set: 1727, number of used features: 32
[LightGBM] [Info] Start training from score 27.470180
Best params: {'max_rounds': np.int64(400), 'max_bins': 32, 'learning_rate': 0.05}
MAE: 14.27445023781182
MSE: 323.6155977385236
RMSE: 17.989318990404378
R2: 0.20509718595858017


In [ ]:
reg = LGBMRegressor(boosting_type='gbdt', objective='regression', random_state=42)

param_distributions = {
    "n_estimators": list(np.arange(50, 401, 50)),
    "max_depth": [-1, 3, 5, 9, 11],   # -1 means no limit
    "num_leaves": [10, 20, 30, 50, 70, 100],
    "learning_rate": [0.01, 0.1, 0.5, 1.0]
    #reg_alpha=0.0,        # L1 regularization
    #reg_lambda=0.0,       # L2 regularization
}

search = RandomizedSearchCV(
    estimator=reg,
    param_distributions=param_distributions,
    n_iter=100, # takes 24 minutes
    cv=5, # changed this because with RepeatedKFold it was on 40 minutes and still didn't finish
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    random_state=0
)

search.fit(X_train, y_train)

best_reg = search.best_estimator_
y_pred = best_reg.predict(X_test)

print("Best params:", search.best_params_)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000823 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4672
[LightGBM] [Info] Number of data points in the train set: 1381, number of used features: 32
[LightGBM] [Info] Start training from score 27.627806
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000467 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4665
[LightGBM] [Info] Number of data points in the train set: 1381, number of used features: 32
[LightGBM] [Info] Start training from score 27.241130
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001594 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4680
[LightGBM] [Info] Number of data points in th

# Explainable Boosting Model

Interpretable boosting model "glassbox". It is a Generalized Additive Model (GAM). Uses tree-based, cyclic gradient boosting with automatic interaction detection, and it is designed to have accuracy comparable to state-of-the-art blackbox models like Random Forest, XGBoost, or LightGBM, while being fully interpretable -> we can visualize each feature’s contribution to predictions (global and local explanations)

In [ ]:
# Note: prefer 'python -m pip install interpret' in the same environment if you ever reinstall packages, rather than '%pip install interpret', to avoid environment mismatch.
# %pip install interpret

In [19]:
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show

In [30]:
reg = ExplainableBoostingRegressor()

reg.fit(X_train, y_train)
y_pred = reg.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))


MAE: 13.357694402814298
MSE: 287.96868551354845
RMSE: 16.969640111491714
R2: 0.2926573376865421


We can also use RandomizedSearchCV with ExplainableBoostingRegressor to tune its hyperparameters, though the default is often quite strong because EBM is already designed to be competitive out of the box.

In [ ]:
reg = ExplainableBoostingRegressor(random_state=42)

param_distributions = {
    "max_rounds": list(np.arange(50, 501, 50)),
    "max_bins": [32, 64, 128, 256],
    "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]
}

search = RandomizedSearchCV(
    estimator=reg,
    param_distributions=param_distributions,
    n_iter=50, # takes 1.6 h
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
    random_state=0
)

search.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",ExplainableBoostingRegressor()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.01, 0.05, ...], 'max_bins': [32, 64, ...], 'max_rounds': [np.int64(50), np.int64(100), ...]}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... 

In [ ]:
best_reg = search.best_estimator_
y_pred = best_reg.predict(X_test)

print("Best params:", search.best_params_)

print("MAE:", mean_absolute_error(y_test, y_pred))
print('MSE:', mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print('R2:', r2_score(y_test, y_pred))

Best params: {'max_rounds': np.int64(150), 'max_bins': 64, 'learning_rate': 0.01}
MAE: 13.333761307865116
MSE: 285.8173788528079
RMSE: 16.906134355694913
R2: 0.29794163093581383


Thanks to EBM we know that for this model, the 5 most important features to predict Total_PCIAT are: computer hours, sds, sex, height, and age. 

In [42]:
ebm_global = best_reg.explain_global(name="EBM")
show(ebm_global)

<!-- http://127.0.0.1:7602/5528796880/ -->

In [ ]:
#Extra:
from interpret.perf import RegressionPerf

# Global explanation
ebm_global = best_reg.explain_global(name="EBM")
show(ebm_global)

# Local explanations for 5 test samples
ebm_local = best_reg.explain_local(X_test[:5], y_test[:5], name="EBM")
show(ebm_local)

# Performance explanation
ebm_perf = RegressionPerf(best_reg).explain_perf(X_test, y_test, name="EBM")
show(ebm_perf)